# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [7]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [8]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [9]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [10]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [11]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [12]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [13]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau).
    # TODO: tau = lam * eps; PGD center in the (eps - tau)-ball around x
    tau = lam * eps
    pgd_eps = eps - tau

    model.eval()
    center = pgd(model, x, y, pgd_eps, norm, steps=5)
    model.train()
    return center, tau


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    # TODO: CE of the negative IBP margin bounds on the small region
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    m = ibp_margin_bounds(model, center, tau, norm, y)

    # z_hat = -m, with z_hat_y = 0
    z_hat = -m
    z_hat = z_hat.scatter(1, y.unsqueeze(1), 0.0)
    return F.cross_entropy(z_hat, y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    # TODO: standard loss for t < 1, then the certified loss with the scheduled radius
    if t < 1:
        return F.cross_entropy(model(x), y)
    eps = eps_scale(t) * EPS[norm] # radius
    lam = LAM[norm]
    return certified_loss(model, x, y, eps, lam, norm)

In [16]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5648  time 0 s
epoch  2  loss 0.2032  time 1 s
epoch  3  loss 0.1519  time 1 s
epoch  4  loss 0.1242  time 1 s
epoch  5  loss 0.1064  time 1 s
epoch  6  loss 0.0898  time 1 s
epoch  7  loss 0.0803  time 1 s
epoch  8  loss 0.0719  time 1 s
epoch  9  loss 0.0634  time 1 s
epoch 10  loss 0.0574  time 1 s
epoch 11  loss 0.0510  time 1 s
epoch 12  loss 0.0476  time 1 s
epoch 13  loss 0.0431  time 1 s
epoch 14  loss 0.0417  time 1 s
epoch 15  loss 0.0361  time 1 s
epoch 16  loss 0.0187  time 2 s
epoch 17  loss 0.0143  time 2 s
epoch 18  loss 0.0132  time 2 s
epoch 19  loss 0.0122  time 2 s
epoch 20  loss 0.0115  time 2 s
epoch  1  loss 0.5843  time 0 s
epoch  2  loss 0.5000  time 1 s
epoch  3  loss 0.4803  time 1 s
epoch  4  loss 0.4883  time 2 s
epoch  5  loss 0.5115  time 2 s
epoch  6  loss 0.5400  time 2 s
epoch  7  loss 0.5776  time 3 s
epoch  8  loss 0.6322  time 3 s
epoch  9  loss 0.6905  time 4 s
epoch 10  loss 0.7649  time 4 s
epoch 11  loss 0.8406  time 5 s
epoch 12

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [17]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    # TODO: per-image max of the two certified losses + eta * KL on the l2-certified subset
    z_hat = {}
    for norm in ['linf', 'l2']:
        eps = eps_scale(t) * EPS[norm]
        center, tau = propagation_region(model, x, y, eps, LAM[norm], norm)

        z_hat[norm] = -ibp_margin_bounds(model, center, tau, norm, y)
        z_hat[norm] = z_hat[norm].scatter(1, y.unsqueeze(1), 0.0)

    # Certified loss per image
    loss_linf = F.cross_entropy(z_hat['linf'], y, reduction='none')
    loss_l2 = F.cross_entropy(z_hat['l2'], y, reduction='none')
    loss_max = torch.maximum(loss_linf, loss_l2).mean()

    # L2-certified subset
    certified = z_hat['l2'] < 0
    certified[torch.arange(len(y), device=y.device), y] = True
    certified = certified.all(dim=1)

    # KL(d_l2 || d_linf)
    loss_ba = torch.zeros((), device=x.device)
    if certified.any():
        linf_logits = z_hat['linf'][certified]
        l2_logits = z_hat['l2'][certified]
        d_linf = F.softmax(linf_logits, dim=1)
        d_l2 = F.softmax(l2_logits, dim=1)
        
        loss_ba = F.kl_div(d_linf.log(), d_l2, reduction='batchmean')
    return loss_max + eta * loss_ba

def gradient_projection(g_n, g_c, beta):
    # TODO: beta * GP(g_n, g_c) + (1 - beta) * g_c for one parameter tensor
    cos = F.cosine_similarity(g_n.flatten(), g_c.flatten(), dim=0)
    gp = torch.zeros_like(g_n)
    if cos > 0:
        gp = cos * g_n
    return beta * gp + (1 - beta) * g_c

In [18]:
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    models[name] = make_model()
    sec = train(models[name], loss_fn, gp_beta=beta)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5843  time 0 s
epoch  2  loss 0.5020  time 1 s
epoch  3  loss 0.5025  time 2 s
epoch  4  loss 0.5369  time 3 s
epoch  5  loss 0.5845  time 4 s
epoch  6  loss 0.6555  time 5 s
epoch  7  loss 0.7468  time 6 s
epoch  8  loss 0.8510  time 7 s
epoch  9  loss 0.9802  time 8 s
epoch 10  loss 1.1197  time 9 s
epoch 11  loss 1.2546  time 10 s
epoch 12  loss 1.3051  time 11 s
epoch 13  loss 1.2767  time 11 s
epoch 14  loss 1.2562  time 12 s
epoch 15  loss 1.2402  time 13 s
epoch 16  loss 1.1887  time 14 s
epoch 17  loss 1.1750  time 15 s
epoch 18  loss 1.1713  time 16 s
epoch 19  loss 1.1683  time 17 s
epoch 20  loss 1.1651  time 18 s
epoch  1  loss 0.5843  time 0 s
epoch  2  loss 0.5006  time 1 s
epoch  3  loss 0.5003  time 2 s
epoch  4  loss 0.5338  time 3 s
epoch  5  loss 0.5839  time 4 s
epoch  6  loss 0.6529  time 5 s
epoch  7  loss 0.7417  time 6 s
epoch  8  loss 0.8452  time 7 s
epoch  9  loss 0.9703  time 8 s
epoch 10  loss 1.1043  time 8 s
epoch 11  loss 1.2424  time 9 

In [36]:
# Seed 1
results = []
for name, loss_fn, beta in [("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
                            ("cure_max_ba", cure_loss, None),
                            ("cure_max_ba_gp", cure_loss, 0.5)]:
    models[name] = make_model()
    sec = train(models[name], loss_fn, gp_beta=beta, seed=1)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5599  time 0 s
epoch  2  loss 0.4914  time 1 s
epoch  3  loss 0.5043  time 2 s
epoch  4  loss 0.5512  time 3 s
epoch  5  loss 0.6084  time 4 s
epoch  6  loss 0.6787  time 5 s
epoch  7  loss 0.7619  time 6 s
epoch  8  loss 0.8667  time 7 s
epoch  9  loss 0.9870  time 8 s
epoch 10  loss 1.1024  time 9 s
epoch 11  loss 1.2274  time 10 s
epoch 12  loss 1.2743  time 11 s
epoch 13  loss 1.2442  time 12 s
epoch 14  loss 1.2266  time 13 s
epoch 15  loss 1.2120  time 14 s
epoch 16  loss 1.1649  time 15 s
epoch 17  loss 1.1542  time 16 s
epoch 18  loss 1.1485  time 17 s
epoch 19  loss 1.1467  time 18 s
epoch 20  loss 1.1447  time 19 s
epoch  1  loss 0.5599  time 0 s
epoch  2  loss 0.4920  time 1 s
epoch  3  loss 0.5046  time 2 s
epoch  4  loss 0.5528  time 3 s
epoch  5  loss 0.6104  time 4 s
epoch  6  loss 0.6800  time 5 s
epoch  7  loss 0.7611  time 6 s
epoch  8  loss 0.8678  time 7 s
epoch  9  loss 0.9929  time 8 s
epoch 10  loss 1.1127  time 9 s
epoch 11  loss 1.2392  time 10

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [39]:
def my_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z_hat = {}
    for norm in ['linf', 'l2']:
        eps = eps_scale(t) * EPS[norm]
        center, tau = propagation_region(model, x, y, eps, LAM[norm], norm)

        # Combine IBP and CROWN bounds
        ibp = ibp_margin_bounds(model, center, tau, norm, y)
        crown = crown_margin_bounds(model, center, tau, norm, y)
        z_hat[norm] = -torch.maximum(ibp, crown)
        z_hat[norm] = z_hat[norm].scatter(1, y.unsqueeze(1), 0.0)

    loss_linf = F.cross_entropy(z_hat['linf'], y, reduction='none')
    loss_l2 = F.cross_entropy(z_hat['l2'], y, reduction='none')
    loss_max = torch.maximum(loss_linf, loss_l2).mean()

    certified = z_hat['l2'] < 0
    certified[torch.arange(len(y), device=y.device), y] = True
    certified = certified.all(dim=1)

    loss_ba = torch.zeros((), device=x.device)
    if certified.any():
        linf_logits = z_hat['linf'][certified]
        l2_logits = z_hat['l2'][certified]
        d_l2 = F.softmax(l2_logits, dim=1)
        log_d_linf = F.log_softmax(linf_logits, dim=1)

        loss_ba = F.kl_div(log_d_linf, d_l2, reduction='batchmean')
    return loss_max + eta * loss_ba

In [41]:
name, loss_fn, beta = ("my_method", my_loss, None)
models[name] = make_model()
sec = train(models[name], loss_fn, gp_beta=beta)
results = dict(evaluate(models[name], name, attacks=True), time=sec)
print_results([results])

epoch  1  loss 0.5843  time 0 s
epoch  2  loss 0.2626  time 11 s
epoch  3  loss 0.2679  time 23 s
epoch  4  loss 0.2945  time 34 s
epoch  5  loss 0.3535  time 45 s
epoch  6  loss 0.4226  time 56 s
epoch  7  loss 0.5000  time 67 s
epoch  8  loss 0.5921  time 79 s
epoch  9  loss 0.7054  time 90 s
epoch 10  loss 0.8204  time 101 s
epoch 11  loss 0.9389  time 112 s
epoch 12  loss 0.9796  time 123 s
epoch 13  loss 0.9628  time 134 s
epoch 14  loss 0.9430  time 146 s
epoch 15  loss 0.9329  time 157 s
epoch 16  loss 0.8793  time 168 s
epoch 17  loss 0.8737  time 179 s
epoch 18  loss 0.8728  time 190 s
epoch 19  loss 0.8704  time 202 s
epoch 20  loss 0.8690  time 213 s
my_method                 clean  95.03  linf  72.48  l2  63.46  union  63.45  pgd_linf  79.65  pgd_l2  80.82  pgd_union  78.92  time/epoch  10.6 s


In [42]:
# Seed 1
name, loss_fn, beta = ("my_method", my_loss, None)
models[name] = make_model()
sec = train(models[name], loss_fn, gp_beta=beta, seed=1)
results = dict(evaluate(models[name], name, attacks=True), time=sec)
print_results([results])

epoch  1  loss 0.5801  time 0 s
epoch  2  loss 0.2633  time 11 s
epoch  3  loss 0.2733  time 23 s
epoch  4  loss 0.3016  time 34 s
epoch  5  loss 0.3523  time 45 s
epoch  6  loss 0.4118  time 56 s
epoch  7  loss 0.5017  time 67 s
epoch  8  loss 0.5884  time 78 s
epoch  9  loss 0.6853  time 90 s
epoch 10  loss 0.8027  time 101 s
epoch 11  loss 0.9256  time 112 s
epoch 12  loss 0.9810  time 123 s
epoch 13  loss 0.9681  time 134 s
epoch 14  loss 0.9558  time 145 s
epoch 15  loss 0.9365  time 156 s
epoch 16  loss 0.8838  time 167 s
epoch 17  loss 0.8776  time 179 s
epoch 18  loss 0.8734  time 190 s
epoch 19  loss 0.8729  time 201 s
epoch 20  loss 0.8718  time 212 s
my_method                 clean  95.35  linf  72.51  l2  62.95  union  62.92  pgd_linf  80.01  pgd_l2  80.82  pgd_union  79.00  time/epoch  10.6 s


In [46]:
# Train CURE for same time (10x epochs)
name, loss_fn, beta = ("cure_max_ba", cure_loss, None)
models[name] = make_model()
sec = train(models[name], loss_fn, gp_beta=beta, epochs=200, verbose=False)
results = dict(evaluate(models[name], name, attacks=True), time=sec)
print_results([results])

cure_max_ba               clean  92.17  linf  67.76  l2  54.23  union  54.21  pgd_linf  79.19  pgd_l2  80.33  pgd_union  78.38  time/epoch   0.9 s


In [38]:
# Ablation
name, loss_fn, beta = ("my_method", my_loss, None)
models[name] = make_model()
sec = train(models[name], loss_fn, gp_beta=beta)
results = dict(evaluate(models[name], name, attacks=True), time=sec)
print_results([results])

epoch  1  loss 0.5599  time 0 s
epoch  2  loss 0.2421  time 15 s
epoch  3  loss 0.2564  time 30 s
epoch  4  loss 0.2950  time 44 s
epoch  5  loss 0.3472  time 59 s
epoch  6  loss 0.4210  time 74 s
epoch  7  loss 0.5037  time 89 s
epoch  8  loss 0.5919  time 104 s
epoch  9  loss 0.7107  time 118 s
epoch 10  loss 0.8264  time 133 s
epoch 11  loss 0.9465  time 148 s
epoch 12  loss 0.9933  time 163 s
epoch 13  loss 0.9644  time 177 s
epoch 14  loss 0.9556  time 192 s
epoch 15  loss 0.9493  time 207 s
epoch 16  loss 0.8876  time 221 s
epoch 17  loss 0.8795  time 236 s
epoch 18  loss 0.8778  time 251 s
epoch 19  loss 0.8752  time 266 s
epoch 20  loss 0.8738  time 280 s
my_method                 clean  95.00  linf  72.40  l2  63.45  union  63.44  pgd_linf  79.23  pgd_l2  80.19  pgd_union  78.32  time/epoch  14.0 s
